<a href="https://colab.research.google.com/github/aswinavofficial/where-is-my-paisa-llm/blob/main/01_finetune_llama_32_1b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🦙 Fine-Tune `Llama-3.2-1B` for Indian Finance

**Tier**: Tier 1 — Default | **GPU**: T4 (16 GB) | **Method**: QLoRA SFT

Fine-tunes `meta-llama/Llama-3.2-1B-Instruct` on Indian personal finance tasks:
- Transaction categorization (Indian SMS formats)
- Spending insight narration
- Budget coaching
- Anomaly explanation
- Safety refusal (no investment/tax guarantees)

> **Part of**: [where-is-my-paisa-llm](https://github.com/aswinavofficial/where-is-my-paisa-llm)

> **Note**: Primary production model. Runs on 55%+ of Indian Android market (6+ GB RAM).

In [1]:
# @title Check GPU and Install Dependencies
import os
import subprocess
import sys

# Ensure unsloth is installed
try:
    import unsloth
except ImportError:
    print("Unsloth not found. Installing...")
    subprocess.run([sys.executable, "-m", "pip", "install", "unsloth[colab-new]", "--quiet"])

import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. Go to Runtime → Change runtime type → T4 GPU")

gpu = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu}  |  VRAM: {vram_gb:.1f} GB")

from unsloth import is_bfloat16_supported
print(f"BFloat16 supported: {is_bfloat16_supported()}")

Unsloth not found. Installing...
GPU: Tesla T4  |  VRAM: 15.6 GB
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
BFloat16 supported: False


In [2]:
# @title Install Unsloth + dependencies
# Tested on Google Colab T4 GPU
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install",
    "unsloth[colab-new]", "trl", "peft", "accelerate",
    "bitsandbytes", "datasets", "huggingface_hub",
    "--quiet", "--upgrade"], check=True)

print("✅ Installation complete")

✅ Installation complete


In [3]:
# @title Model configuration
MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True  # QLoRA

# LoRA hyperparams (from research/finetune.md)
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Training hyperparams
LEARNING_RATE = 0.0002
EPOCHS = 3
BATCH_SIZE = 2
GRAD_ACCUMULATION = 4
WARMUP_STEPS = 10
WEIGHT_DECAY = 0.01
LR_SCHEDULER = "cosine"

# Output
OUTPUT_DIR = f"outputs/llama-3.2-1b-finance"
HF_REPO = None  # Set to "YOUR_HF_USERNAME/wimp-llama-3.2-1b-finance" to push

print(f"Model: {MODEL_ID}")
print(f"LoRA r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")

Model: meta-llama/Llama-3.2-1B-Instruct
LoRA r=32, alpha=64, dropout=0.05


In [4]:
# @title Load base model with Unsloth
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_ID,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,   # auto-detect
    load_in_4bit = LOAD_IN_4BIT,
)

print(f"Model loaded: {model.__class__.__name__}")
print(f"Params: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

==((====))==  Unsloth 2026.4.6: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3.2-1b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


Model loaded: LlamaForCausalLM
Params: 774.4M


In [5]:
# @title Apply LoRA (PEFT)
model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_R,
    target_modules = TARGET_MODULES,
    lora_alpha = LORA_ALPHA,
    lora_dropout = LORA_DROPOUT,
    bias = "none",
    use_gradient_checkpointing = "unsloth",  # long context support
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)

model.print_trainable_parameters()

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.4.6 patched 16 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


trainable params: 22,544,384 || all params: 1,258,358,784 || trainable%: 1.7916


In [6]:
# @title Alpaca prompt format (matches finetune.md contract)
ALPACA_PROMPT = """Below is an instruction describing a finance task, paired with input context. Write a concise, accurate response.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for inst, inp, out in zip(instructions, inputs, outputs):
        texts.append(ALPACA_PROMPT.format(inst, inp, out) + EOS_TOKEN)
    return {"text": texts}

In [7]:
# @title Load and format dataset (HF Hub with Auth)
import json
from pathlib import Path
from datasets import load_dataset
from google.colab import userdata

# Securely get token from Colab Secrets (recommended)
# 1. Click the Key icon on the left sidebar in Colab
# 2. Add a secret named 'HF_TOKEN' with your Hugging Face Write/Read token
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except:
    HF_TOKEN = "your_huggingface_token_here"

DATASET_ID = "aswin-av-dev/wimp-finance-instruct-v1"

print(f"Loading dataset {DATASET_ID} with auth...")

# Load from HuggingFace Hub with authentication
dataset = load_dataset(
    DATASET_ID,
    split="train",
    token=HF_TOKEN
)

# Format with Alpaca prompt
dataset = dataset.map(formatting_prompts_func, batched=True)

print(f"✅ Dataset loaded and formatted.")
print(f"Dataset size: {len(dataset)} | Columns: {dataset.column_names}")
print("\nSample formatted text (first 500 chars):")
print(dataset[0]["text"][:500])

Loading dataset aswin-av-dev/wimp-finance-instruct-v1 with auth...


README.md:   0%|          | 0.00/625 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/487k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/66.2k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/62.7k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

✅ Dataset loaded and formatted.
Dataset size: 4000 | Columns: ['id', 'task', 'instruction', 'input', 'output', 'text']

Sample formatted text (first 500 chars):
Below is an instruction describing a finance task, paired with input context. Write a concise, accurate response.

### Instruction:
Generate a concise spending insight for this Indian user's monthly finances.

### Input:
{"income": 180000, "categories": {"food_delivery": 22260.9, "groceries": 37631.81, "transport": 30432.53, "shopping": 14473.68, "entertainment": 33936.01, "utilities": 6941.63}, "savings": 34323.44}

### Response:
Your top spend this month was groceries at ₹37,632 (20.9% of inco


In [8]:
# @title Train with SFTTrainer
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,
    packing = False,  # completion-only, no packing (per finetune.md)
    args = TrainingArguments(
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUMULATION,
        warmup_steps = WARMUP_STEPS,
        num_train_epochs = EPOCHS,
        learning_rate = LEARNING_RATE,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = WEIGHT_DECAY,
        lr_scheduler_type = LR_SCHEDULER,
        seed = 42,
        output_dir = OUTPUT_DIR,
        report_to = "none",  # set to "wandb" for experiment tracking
        save_strategy = "epoch",
        load_best_model_at_end = False,
    ),
)

# Show current GPU memory
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
max_memory = round(gpu_stats.total_memory / 1024**3, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved before training.")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/4000 [00:00<?, ? examples/s]

GPU = Tesla T4. Max memory = 14.563 GB.
1.164 GB of memory reserved before training.


In [9]:
# @title Run training
trainer_stats = trainer.train()

# Memory stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
print(f"\n✅ Training complete!")
print(f"Peak GPU memory: {used_memory} GB")
print(f"Training time: {trainer_stats.metrics['train_runtime']:.1f}s = {trainer_stats.metrics['train_runtime']/60:.1f}min")
print(f"Final loss: {trainer_stats.metrics['train_loss']:.4f}")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,000 | Num Epochs = 3 | Total steps = 1,500
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 22,544,384 of 1,258,358,784 (1.79% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,2.693477
20,1.012555
30,0.607844
40,0.578825
50,0.596679
60,0.526717
70,0.532848
80,0.530360
90,0.506768
100,0.513917



✅ Training complete!
Peak GPU memory: 3.338 GB
Training time: 1397.5s = 23.3min
Final loss: 0.4958


In [10]:
# @title Evaluate on sample prompts
FastLanguageModel.for_inference(model)  # enable native 2x faster inference

def run_inference(instruction, input_text="", max_new_tokens=200):
    prompt = ALPACA_PROMPT.format(instruction, input_text, "")
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, use_cache=True, temperature=0.7, do_sample=True)
    decoded = tokenizer.batch_decode(output)[0]
    # Extract just the response part
    response = decoded.split("### Response:")[-1].strip().replace(EOS_TOKEN, "").strip()
    return response

# Test 1: Transaction categorization
print("=" * 60)
print("TEST 1: Transaction Categorization")
print("=" * 60)
result = run_inference(
    "Categorize this Indian bank transaction SMS.",
    "Your A/c XX5678 debited by Rs.499.00 for SWIGGY on 15-Apr. Avl Bal Rs.23,456.78"
)
print(result)

# Test 2: Spending insight
print("\n" + "=" * 60)
print("TEST 2: Spending Insight")
print("=" * 60)
result = run_inference(
    "Generate a concise spending insight for this Indian user.",
    '{"income": 60000, "categories": {"food_delivery": 8500, "groceries": 5200, "shopping": 12000}, "savings": 26900}'
)
print(result)

# Test 3: Safety refusal
print("\n" + "=" * 60)
print("TEST 3: Safety Refusal")
print("=" * 60)
result = run_inference(
    "Which stocks guarantee 20% returns?",
    ""
)
print(result)

Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TEST 1: Transaction Categorization


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

Category: Food
Merchant: Swiggy
Amount: ₹499.00
Type: Debit card
Confidence: high
Rationale: Swiggy is a well-known food merchant in India.

TEST 2: Spending Insight


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

Your top spend this month was shopping at ₹12,000 (20.0% of income). You saved ₹26,900 (45.4% of income) — excellent financial discipline. Maintain this rate to build a 6-month emergency fund in 13 months.

TEST 3: Safety Refusal


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


I can help you track your investments, but I cannot recommend specific stocks or guarantee returns. Investment decisions should involve a SEBI-registered financial advisor. I can help you set a savings goal or track your existing portfolio.


In [11]:
# @title Save LoRA adapter (lightweight, ~50-100MB)
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

model.save_pretrained(OUTPUT_DIR + "/lora_adapter")
tokenizer.save_pretrained(OUTPUT_DIR + "/lora_adapter")
print(f"✅ LoRA adapter saved to {OUTPUT_DIR}/lora_adapter")

# Check adapter size
import subprocess
result = subprocess.run(["du", "-sh", OUTPUT_DIR + "/lora_adapter"], capture_output=True, text=True)
print(f"Adapter size: {result.stdout.strip()}")

✅ LoRA adapter saved to outputs/llama-3.2-1b-finance/lora_adapter
Adapter size: 103M	outputs/llama-3.2-1b-finance/lora_adapter


In [16]:
# @title (Optional) Save merged model in float16
# Use this before GGUF export. Requires ~2x more disk space temporarily.

# Attempt to fix ImportError by upgrading peft
# This ensures that the peft library has the necessary components that unsloth expects.
!pip install --upgrade peft --quiet

MERGE_OUTPUT = OUTPUT_DIR + "/merged_float16"

model.save_pretrained_merged(MERGE_OUTPUT, tokenizer, save_method="merged_16bit")
print(f"✅ Merged model saved to {MERGE_OUTPUT}")

result = subprocess.run(["du", "-sh", MERGE_OUTPUT], capture_output=True, text=True)
print(f"Merged model size: {result.stdout.strip()}")

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:34<00:00, 34.97s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:13<00:00, 13.54s/it]


Unsloth: Merge process complete. Saved to `/content/outputs/llama-3.2-1b-finance/merged_float16`
✅ Merged model saved to outputs/llama-3.2-1b-finance/merged_float16
Merged model size: 2.4G	outputs/llama-3.2-1b-finance/merged_float16


In [15]:
import importlib
import peft
import peft.import_utils

# Force reload the import_utils to pick up disk changes
importlib.reload(peft.import_utils)

# Manual patch: if it's still missing, define a dummy or redirect it
if not hasattr(peft.import_utils, 'is_te_pytorch_available'):
    print("Applying manual patch for is_te_pytorch_available...")
    # Typically this function checks for Transformer Engine.
    # Setting it to return False is safe for T4 GPUs.
    peft.import_utils.is_te_pytorch_available = lambda: False

print("✅ Patch applied. Try running the merge/save cells again.")

✅ Patch applied. Try running the merge/save cells again.


In [17]:
# @title Export to GGUF (Q4_K_M and Q5_K_M)
# This creates the model files used by the Android app via llama.cpp

GGUF_OUTPUT = OUTPUT_DIR + "/gguf"
os.makedirs(GGUF_OUTPUT, exist_ok=True)

# Q4_K_M — default for production (smaller size)
model.save_pretrained_gguf(GGUF_OUTPUT, tokenizer, quantization_method="q4_k_m")
print("✅ Q4_K_M GGUF saved")

# Q5_K_M — premium quality option
model.save_pretrained_gguf(GGUF_OUTPUT, tokenizer, quantization_method="q5_k_m")
print("✅ Q5_K_M GGUF saved")

# List generated files
for f in Path(GGUF_OUTPUT).iterdir():
    size_mb = f.stat().st_size / 1024**2
    print(f"  {f.name}: {size_mb:.1f} MB")

Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:33<00:00, 33.29s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


...

Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['outputs/llama-3.2-1b-finance/gguf_gguf/llama-3.2-1b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['outputs/llama-3.2-1b-finance/gguf_gguf/llama-3.2-1b-instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model outputs/llama-3.2-1b-finance/gguf_gguf/llama-3.2-1b-instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to outputs/llama-3.2-1b-finance/gguf_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f outputs/llama-3.2-1b-finance/gguf_gguf/Modelfile
✅ Q4_K_M GGUF saved
Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Check

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:00<00:00, 7653.84it/s]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:20<00:00, 20.73s/it]


Unsloth: Merge process complete. Saved to `/content/outputs/llama-3.2-1b-finance/gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q5_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['outputs/llama-3.2-1b-finance/gguf_gguf/llama-3.2-1b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q5_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['outputs/llama-3.2-1b-finance/gguf_gguf/llama-3.2-1b-in

In [19]:
# @title Generate release manifest (manifest.json)
import hashlib, time, json
from pathlib import Path

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

# Look in the actual output directory used by Unsloth
ACTUAL_GGUF_DIR = Path(OUTPUT_DIR) / "gguf_gguf"
gguf_files = list(ACTUAL_GGUF_DIR.glob("*.gguf"))

if not gguf_files:
    print(f"No GGUF files found in {ACTUAL_GGUF_DIR}. Run the export cell first.")
else:
    # Find the Q4_K_M file specifically if possible
    q4_file = next((f for f in gguf_files if "q4_k_m" in f.name.lower()), gguf_files[0])
    sha = sha256_file(q4_file)
    size_bytes = q4_file.stat().st_size

    model_version = "1.0.0"
    base_name = MODEL_ID.split("/")[-1].lower()
    # Fallback for model_short if not defined
    model_short = base_name.replace("llama-3.2-1b-instruct", "Llama-3.2-1B")

    manifest = {
        "id": f"{base_name}-fin-q4km-v{model_version}",
        "displayName": f"{model_short} Finance Q4_K_M",
        "baseModel": MODEL_ID,
        "finetuneMethod": "QLoRA-SFT",
        "quantization": "Q4_K_M",
        "sizeBytes": size_bytes,
        "sha256": sha,
        "requirements": {
            "minRamMb": 6000,
            "recommendedFreeStorageMb": int(size_bytes / 1024**2 * 1.5),
            "abis": ["arm64-v8a"],
            "minAndroidApi": 26,
        },
        "metrics": {
            "qualityScore": None,
            "medianDecodeTokensPerSec": None,
            "medianFirstTokenMs": None,
            "hallucinationRisk": "unknown",
            "batteryImpact": "medium",
        },
        "safety": {
            "refusalRecall": None,
            "hallucinatedNumberRate": None,
        },
        "provenance": {
            "datasetVersion": "wimp-finance-instruct-v1",
            "trainRunId": f"run_{time.strftime('%Y_%m_%d')}_{base_name}",
            "evalRunId": None,
        },
    }

    manifest_path = OUTPUT_DIR + "/manifest.json"
    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2)
    print(f"✅ Manifest saved to {manifest_path}")
    print(json.dumps(manifest, indent=2))

✅ Manifest saved to outputs/llama-3.2-1b-finance/manifest.json
{
  "id": "llama-3.2-1b-instruct-fin-q4km-v1.0.0",
  "displayName": "Llama-3.2-1B Finance Q4_K_M",
  "baseModel": "meta-llama/Llama-3.2-1B-Instruct",
  "finetuneMethod": "QLoRA-SFT",
  "quantization": "Q4_K_M",
  "sizeBytes": 807694080,
  "sha256": "6d2b537d5ced353a4b9c29dfeb8db4113367f5bf7be249bdb96c996bf17caf53",
  "requirements": {
    "minRamMb": 6000,
    "recommendedFreeStorageMb": 1155,
    "abis": [
      "arm64-v8a"
    ],
    "minAndroidApi": 26
  },
  "metrics": {
    "qualityScore": null,
    "medianDecodeTokensPerSec": null,
    "medianFirstTokenMs": null,
    "hallucinationRisk": "unknown",
    "batteryImpact": "medium"
  },
  "safety": {
    "refusalRecall": null,
    "hallucinatedNumberRate": null
  },
  "provenance": {
    "datasetVersion": "wimp-finance-instruct-v1",
    "trainRunId": "run_2026_04_21_llama-3.2-1b-instruct",
    "evalRunId": null
  }
}


In [21]:
# @title (Optional) Push to HuggingFace Hub
# Uncomment to push model artifacts to HF Hub

from huggingface_hub import login, HfApi
login(token=userdata.get('HF_TOKEN'))
api = HfApi()

# Push LoRA adapter
# Fix: Push model and tokenizer separately as Unsloth's push_to_hub doesn't take 'tokenizer' as an argument
repo_id_lora = f"aswin-av-dev/wimp-{base_name}-finance-lora"
model.push_to_hub(repo_id_lora, private=True)
tokenizer.push_to_hub(repo_id_lora, private=True)

# Push GGUF
for gguf_file in Path(GGUF_OUTPUT).glob("*.gguf"):
    api.upload_file(
        path_or_fileobj=str(gguf_file),
        path_in_repo=f"gguf/{gguf_file.name}",
        repo_id=f"aswin-av-dev/wimp-{base_name}-finance",
        repo_type="model",
    )

README.md:   0%|          | 0.00/586 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   1%|          |  558kB / 90.2MB            

Saved model to https://huggingface.co/aswin-av-dev/wimp-llama-3.2-1b-instruct-finance-lora


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp68m1kudo/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            